In [1]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "src" / "utils.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.utils import load_csv, data_path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

df = load_csv('data/joined_with_tt_scores.csv')

exclude_cols = ['Unnamed: 0.1', 'Unnamed: 0', 'rating', 'title', 'tag', 'rating_timestamp', 'tag_timestamp', 'userId', 'movieId', '(no genres listed)', 'two_tower_score']

genre_cols = [col for col in df.columns if col not in exclude_cols]

In [2]:
df_weighted = df.copy()

for col in genre_cols:
    df_weighted[col] = df_weighted[col] * df_weighted['rating']

user_profile = df_weighted.groupby('userId')[genre_cols].mean()

user_avg = df.groupby('userId')['rating'].mean()

user_profile = user_profile.apply(
    lambda row: row.fillna(user_avg[row.name]),
    axis=1
)

In [3]:
df_model = df.merge(user_profile, on='userId', suffixes=('', '_user_profile'))

for col in genre_cols:
    df_model[f'{col}_interaction'] = df_model[col] * df_model[f'{col}_user_profile']

feature_cols = [f'{col}_interaction' for col in genre_cols] + ['two_tower_score']

X = df_model[feature_cols]
y = df_model['rating'] >= 4.0

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

print("Logistic F1:", f1_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

importance = pd.Series(log_model.coef_[0], index=feature_cols)
importance = importance.sort_values(key=abs, ascending=False)

print(importance.head(15))

Logistic F1: 0.6864804241435563
              precision    recall  f1-score   support

       False       0.70      0.73      0.71     10525
        True       0.70      0.67      0.69     10011

    accuracy                           0.70     20536
   macro avg       0.70      0.70      0.70     20536
weighted avg       0.70      0.70      0.70     20536

two_tower_score            1.436385
Drama_interaction          0.149502
Comedy_interaction         0.098275
Documentary_interaction    0.075165
Sci-Fi_interaction         0.075073
Musical_interaction        0.061106
Crime_interaction          0.054816
Romance_interaction        0.049905
Adventure_interaction      0.048411
Horror_interaction         0.047414
War_interaction            0.046674
Action_interaction         0.043711
Film-Noir_interaction      0.038766
Western_interaction        0.029943
Thriller_interaction       0.026653
dtype: float64


In [19]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("KNN F1:", f1_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

KNN F1: 0.6702497628833386
              precision    recall  f1-score   support

       False       0.68      0.75      0.72     10525
        True       0.71      0.64      0.67     10011

    accuracy                           0.70     20536
   macro avg       0.70      0.69      0.69     20536
weighted avg       0.70      0.70      0.69     20536



In [20]:
#Recommendation Engine!!!! 

movies_df = df[['movieId', 'title'] + genre_cols + ['two_tower_score']].drop_duplicates()

def recommend_movies(user_id, top_n=10):
    
    # Get user preferences
    user_prefs = user_profile.loc[user_id]
    
    temp_df = movies_df.copy()
    
    # Create interaction features
    for col in genre_cols:
        temp_df[f'{col}_interaction'] = temp_df[col] * user_prefs[col]
    
    X_pred = temp_df[[f'{col}_interaction' for col in genre_cols] + ['two_tower_score']]
    
    # Scale
    X_pred_scaled = scaler.transform(X_pred)
    
    # Predict probability
    scores = knn.predict_proba(X_pred_scaled)[:, 1]
    
    temp_df['score'] = scores
    
    # Remove movies already seen
    watched = df[df['userId'] == user_id]['movieId']
    temp_df = temp_df[~temp_df['movieId'].isin(watched)]
    
    # Return top movies
    return temp_df.sort_values('score', ascending=False).head(top_n)

In [21]:
recommend_movies(user_id=1, top_n=10)

,movieId,title,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,...,Horror_interaction,IMAX_interaction,Musical_interaction,Mystery_interaction,Romance_interaction,Sci-Fi_interaction,Thriller_interaction,War_interaction,Western_interaction,score
102675,168252,Logan (2017),True,False,False,False,False,False,False,False,...,0.0,0.0,0.0,0.0,0.000000,0.728448,0.0,0.0,0.0,1.0
32769,1247,"Graduate, The (1967)",False,False,False,False,True,False,False,True,...,0.0,0.0,0.0,0.0,0.482759,0.000000,0.0,0.0,0.0,1.0
85439,80549,Easy A (2010),False,False,False,False,True,False,False,False,...,0.0,0.0,0.0,0.0,0.482759,0.000000,0.0,0.0,0.0,1.0
32717,912,Casablanca (1942),False,False,False,False,False,False,False,True,...,0.0,0.0,0.0,0.0,0.482759,0.000000,0.0,0.0,0.0,1.0
85443,318,"Shawshank Redemption, The (1994)",False,False,False,False,False,True,False,True,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0
32728,994,Big Night (1996),False,False,False,False,True,False,False,True,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0
32729,1041,Secrets & Lies (1996),False,False,False,False,False,False,False,True,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0
32731,1079,"Fish Called Wanda, A (1988)",False,False,False,False,True,True,False,False,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0
32743,1172,Cinema Paradiso (Nuovo cinema Paradiso) (1989),False,False,False,False,False,False,False,True,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,1.0
85483,589,Terminator 2: Judgment Day (1991),True,False,False,False,False,False,False,False,...,0.0,0.0,0.0,0.0,0.000000,0.728448,0.0,0.0,0.0,1.0
